In [ ]:
import pandas as pd
import requests
from datetime import date, datetime, time, timedelta, timezone
from zoneinfo import ZoneInfo

In [ ]:
timezone_wib = ZoneInfo("Asia/Jakarta")
today_wib = datetime.now(timezone_wib).date()
start_wib = datetime(2026, 8, 1, tzinfo=timezone_wib)
end_date_wib = today_wib - timedelta(days=1)
end_wib = datetime.combine(end_date_wib, time(23, 59, 59, 999000), tzinfo=timezone_wib)

start_date = start_wib.astimezone(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")
end_date = end_wib.astimezone(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
params = {
    "format": "geojson",
    "starttime": start_date,
    "endtime": end_date,
    "eventtype": "earthquake",
    "orderby": "time-asc",
    "limit": 20000,
}

response = requests.get(url, params=params, timeout=120)
response.raise_for_status()
data = response.json()

print(f"Periode WIB: 2026-08-01 00:00:00 sampai {end_date_wib}")
print(f"Query UTC: {start_date} sampai {end_date}")
print(f"Jumlah data: {len(data['features'])}")

In [ ]:
df_usgs = pd.json_normalize(data["features"], sep="_")

coordinates = df_usgs["geometry_coordinates"].apply(pd.Series)
df_usgs["longitude"] = coordinates[0]
df_usgs["latitude"] = coordinates[1]
df_usgs["depth_km"] = coordinates[2]
df_usgs = df_usgs.drop(columns=["geometry_coordinates"])

df_usgs = df_usgs.rename(columns={
    "type": "feature_type",
    "id": "event_id",
    "properties_mag": "magnitude",
    "properties_magType": "magnitude_type",
    "properties_place": "place",
    "properties_time": "event_time",
    "properties_updated": "updated_at",
    "properties_tz": "timezone_offset_minutes",
    "properties_url": "event_url",
    "properties_detail": "detail_url",
    "properties_felt": "felt_reports",
    "properties_cdi": "cdi",
    "properties_mmi": "mmi",
    "properties_alert": "alert",
    "properties_status": "status",
    "properties_tsunami": "tsunami",
    "properties_sig": "significance",
    "properties_net": "network",
    "properties_code": "code",
    "properties_ids": "related_event_ids",
    "properties_sources": "sources",
    "properties_types": "product_types",
    "properties_nst": "station_count",
    "properties_dmin": "minimum_distance",
    "properties_rms": "rms_travel_time",
    "properties_gap": "azimuthal_gap",
    "properties_type": "event_type",
    "properties_title": "title",
})

df_usgs["event_time"] = pd.to_datetime(df_usgs["event_time"], unit="ms", utc=True)
df_usgs["updated_at"] = pd.to_datetime(df_usgs["updated_at"], unit="ms", utc=True)
df_usgs["ingest_at"] = pd.Timestamp.now(tz="UTC")
df_usgs

In [ ]:
df_usgs.info()

In [ ]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
from pathlib import Path
import os

path_env = "/home/dentawinardi/data-engineering-learning/supabase_denta.env"
load_dotenv(dotenv_path=path_env, override=True)
database_url = os.getenv("POSTGRES_URL")
engine_pg = create_engine(database_url)

In [ ]:
from datetime import datetime

try:
    df_usgs.to_sql(
        "usgs_earthquakes_raw",
        engine_pg,
        schema="public",
        if_exists="replace",
        index=False,
        chunksize=500,
        method="multi",
    )
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"Data inserted successfully on PostgreSQL {now}!")
except Exception as e:
    print(f"Error occurred while inserting data: {str(e)}")